# LLM Attribution - full workflow

End-to-end walkthrough: explore a labeled corpus of model-written passages, train the linear TF-IDF baseline, evaluate it with leakage-safe grouped cross-validation, and fine-tune the DeBERTa-v3 transformer track.

**Input:** a CSV with one row per passage and columns `text` (the passage), `model` (which LLM wrote it), and `prompt_id` (which prompt produced it - used for grouped CV).

## 1. Setup and load

In [ ]:
# Copyright 2026 Parth Maniar
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = "../data/corpus.csv"  # text, model, prompt_id
df = pd.read_csv(DATA)
print(df.shape)
df.head()

## 2. Data exploration

Class balance first - attribution metrics only mean something relative to the class prior.

In [ ]:
# Copyright 2026 Parth Maniar
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

counts = df["model"].value_counts().sort_values()
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(counts.index, counts.values, color="#0d9488")
ax.set_xlabel("labeled passages")
ax.set_title("Passages per model")
plt.show()
print(f"{df['model'].nunique()} models, {len(df)} passages, "
      f"min/max per class: {counts.min()}/{counts.max()}")

Passage length varies by orders of magnitude, which is why character-level features matter.

In [ ]:
# Copyright 2026 Parth Maniar
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

lens = df["text"].str.len().clip(upper=20000)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(lens, bins=80, color="#1e3a8a")
ax.set_xlabel("passage length (characters)")
ax.set_ylabel("passages")
plt.show()

## 3. Leakage-safe evaluation protocol

Every prompt's passages stay inside a single fold. Scoring on *seen* prompts would mostly measure context memorization, not model style. Balanced accuracy treats all 18 models equally.

In [ ]:
# Copyright 2026 Parth Maniar
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

from llm_attribution.evaluate import grouped_cv_score
from llm_attribution.linear import build_linear_baseline

texts = df["text"].tolist()
labels = df["model"].to_numpy()
groups = df["prompt_id"].to_numpy() if "prompt_id" in df else df.index.to_numpy()

mean, std, folds = grouped_cv_score(
    build_linear_baseline, texts, labels, groups, n_splits=5
)
print("per-fold:", ", ".join(f"{s:.4f}" for s in folds))
print(f"grouped CV balanced accuracy: {mean:.4f} +- {std:.4f}")
print(f"random chance: {1/df['model'].nunique():.4f}")

## 4. Linear TF-IDF baseline - final fit

Word 1-2 grams plus char_wb 3-5 grams, multinomial logistic regression. Trains in minutes on a CPU.

In [ ]:
# Copyright 2026 Parth Maniar
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

clf = build_linear_baseline(C=4.0)
clf.fit(texts, labels)
print("vocabulary size:", clf.named_steps["features"].transform(texts[:1]).shape[1])

## 5. Transformer track - DeBERTa-v3 fine-tune

Deeper sequential signals via KerasHub (torch backend). Needs a GPU and a local KerasHub preset directory; runs fully offline from there. Batch size 8 at sequence length 256 fits a 16 GB card.

In [ ]:
# Copyright 2026 Parth Maniar
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

from llm_attribution.transformer import build_classifier, fine_tune, predict_labels

PRESET_DIR = "../models/deberta_v3_small_en"  # local KerasHub preset
classes = sorted(df["model"].unique())
label_ids = np.array([classes.index(m) for m in df["model"]], dtype="int32")

clf_t = build_classifier(PRESET_DIR, num_classes=len(classes))
fine_tune(clf_t, texts, label_ids, epochs=2, batch_size=8)

pred = predict_labels(clf_t, df["text"].tolist()[:8], classes)
list(zip(df["model"].tolist()[:8], pred))

## 6. Takeaways

- Character n-grams are the strongest cheap signal: models differ in formatting and punctuation habits more than in vocabulary.
- Prompt-grouped CV is the honest protocol; ungrouped splits overstate accuracy by a wide margin.
- The transformer track adds capacity at ~10x compute; compare against the linear baseline under the same grouped protocol before adopting it.